# CrewAI Content Pipeline Crew

**Project:** Multi-Agent Content Creation & Repurposing Pipeline  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **5-agent content production crew** using [CrewAI](https://docs.crewai.com/). Starting from a single topic, the crew researches, writes, edits, optimizes for SEO, and repurposes content across formats — all coordinated automatically.

### The Agent Roster

| # | Agent | Role | What It Does |
|---|-------|------|-------------|
| 1 | **Researcher** | Deep-Dive Investigator | Gathers facts, statistics, and source material on the topic |
| 2 | **Writer** | Content Creator | Transforms research into a polished long-form blog post |
| 3 | **Editor** | Quality Gatekeeper | Reviews for clarity, grammar, structure, and tone consistency |
| 4 | **SEO Checker** | Search Optimizer | Audits the article for keyword strategy, meta tags, and headings |
| 5 | **Repurposer** | Format Transformer | Converts the article into Twitter threads, LinkedIn posts, and email newsletters |

### How Task Delegation Works in CrewAI

```
Topic Input
    │
    ▼
┌─────────────┐   research     ┌─────────────┐   draft      ┌─────────────┐
│  Researcher  │──────────────>│   Writer     │────────────>│   Editor    │
└─────────────┘               └─────────────┘              └─────────────┘
                                                                  │
                                                            edited article
                                                                  │
                                                    ┌─────────────┴──────────────┐
                                                    │                            │
                                                    ▼                            ▼
                                             ┌─────────────┐           ┌──────────────┐
                                             │ SEO Checker  │           │  Repurposer  │
                                             └─────────────┘           └──────────────┘
                                                    │                            │
                                                    ▼                            ▼
                                              SEO-optimized              Multi-format
                                                article                   content pack
```

**Sequential delegation** means each agent's output automatically becomes the next agent's input. The Crew orchestrator handles all context passing — agents don't need to know about each other.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

We use **Ollama** running locally — no API keys required. All agents share the same LLM instance, but their different roles/backstories produce specialized behavior.

In [ ]:
# Configure the local LLM via Ollama
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

### Agent Architecture in CrewAI

Each agent is defined by three properties that together create a **"persona"** for the LLM:

| Property | Purpose | Analogy |
|----------|---------|---------|
| `role` | Job title — anchors identity | "You are a ___" |
| `goal` | Objective — drives decision-making | "Your mission is to ___" |
| `backstory` | Expertise context — shapes reasoning | "You have spent 10 years ___" |

The **backstory** is the most powerful lever — it acts as a system prompt that shapes:
- What the agent considers important
- How it structures its output
- What frameworks and terminology it uses
- Its quality standards and self-critique behavior

### 3.1 Researcher Agent

**Delegation role:** First in the pipeline — produces the raw material that all downstream agents build on. Quality here determines quality everywhere.

**Key design choice:** `allow_delegation=False` ensures the researcher does its own work rather than asking other agents for help.

In [ ]:
researcher = Agent(
    role="Senior Research Analyst",
    goal=(
        "Conduct deep, thorough research on the given topic. Gather key facts, "
        "statistics, expert opinions, recent developments, and contrarian viewpoints. "
        "Organize findings into a structured research brief that a writer can use."
    ),
    backstory=(
        "You are an investigative research analyst with 12 years of experience at "
        "leading media organizations. You have a talent for finding non-obvious angles, "
        "verifying claims, and structuring complex information. You always provide "
        "specific data points, cite trends with years, and flag areas where data is "
        "uncertain. Your research briefs are known for being comprehensive yet concise, "
        "with clear sections for key findings, supporting evidence, and knowledge gaps."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {researcher.role}")

### 3.2 Writer Agent

**Delegation role:** Receives the research brief and transforms raw data into engaging long-form content. The writer's output is the "first draft" that flows to the editor.

In [ ]:
writer = Agent(
    role="Senior Content Writer",
    goal=(
        "Transform research findings into an engaging, well-structured, long-form "
        "blog post (1500-2000 words). The article should be informative yet accessible, "
        "with a compelling hook, clear structure, and strong conclusion."
    ),
    backstory=(
        "You are an award-winning content writer who has written for publications like "
        "TechCrunch, Wired, and Harvard Business Review. You specialize in making "
        "complex topics accessible without dumbing them down. Your writing style is "
        "conversational yet authoritative. You always use the inverted pyramid structure, "
        "include concrete examples, and end with actionable takeaways. You write in "
        "markdown with proper headings (H2, H3), bullet points, and bold key terms."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {writer.role}")

### 3.3 Editor Agent

**Delegation role:** The **quality gate** — catches errors, improves clarity, and ensures the piece meets professional standards before it moves to optimization and repurposing.

In [ ]:
editor = Agent(
    role="Senior Content Editor",
    goal=(
        "Review and improve the blog post for clarity, grammar, flow, factual accuracy, "
        "and tone consistency. Fix structural issues, tighten prose, ensure logical "
        "transitions, and verify that claims are supported by the research."
    ),
    backstory=(
        "You are a senior editor with 15 years of experience at major digital publications. "
        "You have a sharp eye for passive voice, weak transitions, unsupported claims, and "
        "structural imbalances. You follow the Chicago Manual of Style and prioritize "
        "readability (Flesch-Kincaid Grade 8-10). You return the FULL edited article with "
        "your changes applied, not just a list of suggestions. You also add editorial notes "
        "in [brackets] for anything the writer should reconsider."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {editor.role}")

### 3.4 SEO Checker Agent

**Delegation role:** Receives the edited article and audits it for search engine optimization. Produces both an annotated version and a separate SEO scorecard.

**Why a separate SEO agent?** Combining editing and SEO in one agent dilutes both. A dedicated SEO agent can focus entirely on keyword density, heading structure, meta descriptions, and internal linking without compromising editorial quality.

In [ ]:
seo_checker = Agent(
    role="SEO Content Strategist",
    goal=(
        "Audit the edited article for SEO best practices and produce an optimized "
        "version along with a detailed SEO scorecard. Ensure the article will rank "
        "well for the target topic without sacrificing readability."
    ),
    backstory=(
        "You are an SEO strategist who has helped 100+ articles reach page-one rankings. "
        "You understand on-page SEO deeply: keyword placement (title, H2s, first paragraph, "
        "meta description), semantic keyword clusters, heading hierarchy (single H1, logical "
        "H2/H3 nesting), internal linking opportunities, and content length benchmarks. "
        "You analyze keyword density without keyword stuffing, suggest meta titles (under "
        "60 chars) and meta descriptions (under 160 chars), and check for image alt-text "
        "placeholders. You produce a scorecard rating each SEO dimension 1-10."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {seo_checker.role}")

### 3.5 Repurposer Agent

**Delegation role:** The **final agent** — takes the polished article and transforms it into multiple content formats for different platforms. This maximizes the ROI of a single research + writing effort.

In [ ]:
repurposer = Agent(
    role="Content Repurposing Specialist",
    goal=(
        "Transform the finished blog post into multiple content formats: "
        "a Twitter/X thread (8-12 tweets), a LinkedIn post (300 words), "
        "and an email newsletter edition. Each format should be optimized "
        "for its platform while preserving key messages."
    ),
    backstory=(
        "You are a multi-platform content strategist who specializes in atomizing "
        "long-form content into platform-native formats. You understand that each "
        "platform has different norms: Twitter rewards punchy, numbered threads with "
        "hooks; LinkedIn favors professional storytelling with line breaks; email "
        "newsletters need subject lines, preview text, and clear CTAs. You never "
        "just truncate — you restructure and rewrite for each medium. You always "
        "include platform-specific formatting (thread numbering, emoji usage, etc)."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {repurposer.role}")

### Agent Roster Summary

In [ ]:
agents = [researcher, writer, editor, seo_checker, repurposer]

print("=" * 70)
print("CONTENT CREW ROSTER")
print("=" * 70)
for i, agent in enumerate(agents, 1):
    print(f"\n[{i}] {agent.role}")
    print(f"    Goal: {agent.goal[:90]}...")
    print(f"    Delegation: {'Enabled' if agent.allow_delegation else 'Disabled'}")
print("\n" + "=" * 70)
print(f"\nTotal agents: {len(agents)}")

## 4. Define Tasks & Delegation Chain

### How CrewAI Delegates Tasks

In **sequential mode**, CrewAI delegates tasks using a **chain pattern**:

1. The Crew takes the task list in order
2. Task 1 is assigned to its designated agent
3. When Task 1 completes, its output becomes **context** for Task 2
4. This continues until all tasks are done
5. The final task's output becomes the crew's result

**Key insight:** Each agent only sees what it needs. The researcher doesn't see the editor's work. The editor doesn't see the SEO audit. This mirrors real-world content teams where each role has a clear handoff.

### 4.1 Configure the Content Topic

The pipeline is parameterized — change the topic and re-run to produce a full content package on any subject.

In [ ]:
# =====================================================
# CONFIGURE YOUR CONTENT TOPIC HERE
# =====================================================
TOPIC = "The Rise of Small Language Models: Why Smaller AI Models Are Winning in Production"
AUDIENCE = "Tech-savvy professionals, CTOs, ML engineers, and startup founders"
TONE = "Authoritative yet conversational — backed by data, easy to read"
TARGET_KEYWORDS = ["small language models", "SLM", "edge AI", "model efficiency", "on-device AI"]
# =====================================================

print(f"Topic: {TOPIC}")
print(f"Audience: {AUDIENCE}")
print(f"Tone: {TONE}")
print(f"Target Keywords: {', '.join(TARGET_KEYWORDS)}")

### 4.2 Task 1 — Research

**Delegated to:** Researcher  
**Input:** Topic + audience definition  
**Output:** Structured research brief  
**Downstream consumers:** Writer (directly), Editor (indirectly via writer's draft)

In [ ]:
task_research = Task(
    description=(
        f"Research the following topic thoroughly:\n"
        f"TOPIC: {TOPIC}\n"
        f"TARGET AUDIENCE: {AUDIENCE}\n\n"
        "Your research brief MUST include:\n"
        "1. **Key Facts & Statistics**: At least 8-10 specific data points with context\n"
        "2. **Recent Developments**: What happened in the last 12 months\n"
        "3. **Expert Perspectives**: Contrasting viewpoints from industry leaders\n"
        "4. **Case Studies**: 2-3 real-world examples or company stories\n"
        "5. **Counterarguments**: What critics say and why they might be right\n"
        "6. **Knowledge Gaps**: Areas where data is inconclusive or evolving\n"
        "7. **Suggested Angles**: 3 possible angles the writer could take\n"
    ),
    expected_output=(
        "A structured research brief with numbered sections covering facts, "
        "developments, expert views, case studies, counterarguments, gaps, "
        "and suggested angles. Each fact should include context and sourcing notes."
    ),
    agent=researcher,
)

print(f"Task created: Research -> {task_research.agent.role}")

### 4.3 Task 2 — Write

**Delegated to:** Writer  
**Input:** Research brief (from Task 1)  
**Output:** 1500-2000 word blog post draft  
**Delegation pattern:** The writer receives the researcher's full output as context

In [ ]:
task_write = Task(
    description=(
        f"Using the research brief provided, write a compelling blog post.\n\n"
        f"TOPIC: {TOPIC}\n"
        f"AUDIENCE: {AUDIENCE}\n"
        f"TONE: {TONE}\n"
        f"TARGET KEYWORDS: {', '.join(TARGET_KEYWORDS)}\n\n"
        "Requirements:\n"
        "1. **Hook**: Open with a surprising fact, bold claim, or compelling question\n"
        "2. **Structure**: Use markdown with H2 and H3 headings, bullet points, bold terms\n"
        "3. **Length**: 1500-2000 words\n"
        "4. **Examples**: Include at least 2 real-world examples from the research\n"
        "5. **Data**: Weave in statistics naturally (don't just list them)\n"
        "6. **Conclusion**: End with 3 actionable takeaways\n"
        "7. **Keywords**: Naturally incorporate the target keywords throughout\n"
    ),
    expected_output=(
        "A complete blog post in markdown format, 1500-2000 words, with proper "
        "heading hierarchy, embedded data points, real examples, and actionable "
        "takeaways. Ready for editorial review."
    ),
    agent=writer,
)

print(f"Task created: Write -> {task_write.agent.role}")

### 4.4 Task 3 — Edit

**Delegated to:** Editor  
**Input:** Blog post draft (from Task 2), plus context from research (Task 1)  
**Output:** Polished, publication-ready article  
**Delegation pattern:** The editor sees the writer's draft AND can cross-reference the original research

In [ ]:
task_edit = Task(
    description=(
        "Edit the blog post draft for publication quality.\n\n"
        "Your editing pass MUST address:\n"
        "1. **Grammar & Style**: Fix errors, passive voice, weak verbs, filler words\n"
        "2. **Structure**: Ensure logical flow, smooth transitions between sections\n"
        "3. **Fact-checking**: Verify claims against the original research brief\n"
        "4. **Readability**: Target Flesch-Kincaid Grade Level 8-10\n"
        "5. **Consistency**: Uniform tone, terminology, and formatting throughout\n"
        "6. **Tightening**: Remove redundancy, cut unnecessary words (aim for -10%)\n"
        "7. **Headlines**: Ensure H2s are compelling and descriptive\n\n"
        "IMPORTANT: Return the FULL edited article, not just a list of changes. "
        "Add [EDITOR NOTE: ...] for any suggestions that require the writer's judgment."
    ),
    expected_output=(
        "The complete edited blog post with all corrections applied inline. "
        "Improved clarity, flow, and readability. Any editorial notes in brackets."
    ),
    agent=editor,
)

print(f"Task created: Edit -> {task_edit.agent.role}")

### 4.5 Task 4 — SEO Audit

**Delegated to:** SEO Checker  
**Input:** Edited article (from Task 3)  
**Output:** SEO-optimized article + scorecard  
**Delegation pattern:** SEO analysis happens after editing to avoid re-introducing issues

In [ ]:
task_seo = Task(
    description=(
        f"Perform an SEO audit on the edited blog post.\n\n"
        f"Target keywords: {', '.join(TARGET_KEYWORDS)}\n\n"
        "Your audit MUST cover:\n"
        "1. **Title Tag**: Suggest an SEO-optimized title (under 60 characters)\n"
        "2. **Meta Description**: Write a compelling meta description (under 160 chars)\n"
        "3. **Heading Analysis**: Check H1/H2/H3 hierarchy and keyword inclusion\n"
        "4. **Keyword Density**: Assess primary keyword frequency (target: 1-2%)\n"
        "5. **First Paragraph**: Does it include the primary keyword?\n"
        "6. **Internal Linking**: Suggest 3-5 internal link opportunities\n"
        "7. **Readability Score**: Estimate Flesch-Kincaid grade\n"
        "8. **Content Length**: Compare to top-ranking articles for this topic\n"
        "9. **Image Suggestions**: Recommend image placements with alt-text\n\n"
        "Produce TWO outputs:\n"
        "A) The article with SEO improvements applied inline\n"
        "B) A scorecard rating each SEO dimension 1-10 with improvement notes"
    ),
    expected_output=(
        "Part A: The SEO-optimized article with changes applied. "
        "Part B: An SEO scorecard table rating each dimension 1-10."
    ),
    agent=seo_checker,
)

print(f"Task created: SEO Audit -> {task_seo.agent.role}")

### 4.6 Task 5 — Repurpose

**Delegated to:** Repurposer  
**Input:** SEO-optimized article (from Task 4)  
**Output:** Multi-format content pack (Twitter thread, LinkedIn post, email newsletter)  
**Delegation pattern:** This is the final task — its output is the crew's final result

In [ ]:
task_repurpose = Task(
    description=(
        "Repurpose the blog post into three platform-specific formats.\n\n"
        "Create ALL THREE of the following:\n\n"
        "1. **Twitter/X Thread** (8-12 tweets):\n"
        "   - Start with a hook tweet that creates curiosity\n"
        "   - Number each tweet (1/N format)\n"
        "   - End with a CTA linking to the full article\n"
        "   - Use 1-2 relevant emojis per tweet (not excessive)\n"
        "   - Each tweet under 280 characters\n\n"
        "2. **LinkedIn Post** (250-350 words):\n"
        "   - Professional storytelling format\n"
        "   - Use line breaks for readability (LinkedIn algorithm favors this)\n"
        "   - Include a personal angle or opinion\n"
        "   - End with a question to drive engagement\n"
        "   - Add 3-5 relevant hashtags\n\n"
        "3. **Email Newsletter** (400-500 words):\n"
        "   - Compelling subject line (under 50 chars) + preview text (under 90 chars)\n"
        "   - Conversational, like writing to a friend\n"
        "   - Summarize key insights (not the full article)\n"
        "   - Include a clear CTA to read the full post\n"
        "   - P.S. line with a bonus insight or teaser\n"
    ),
    expected_output=(
        "Three clearly separated content pieces: (1) Twitter/X thread with 8-12 "
        "numbered tweets, (2) LinkedIn post with hashtags, (3) Email newsletter "
        "with subject line, preview text, body, and P.S."
    ),
    agent=repurposer,
)

print(f"Task created: Repurpose -> {task_repurpose.agent.role}")

### Task Delegation Chain

In [ ]:
tasks = [task_research, task_write, task_edit, task_seo, task_repurpose]
task_names = ["Research", "Write", "Edit", "SEO Audit", "Repurpose"]

print("TASK DELEGATION CHAIN")
print("=" * 65)
for i, (name, task) in enumerate(zip(task_names, tasks), 1):
    status = "FINAL OUTPUT" if i == len(tasks) else f"delegates to -> Task {i+1}"
    print(f"  Task {i}: {name}")
    print(f"    Agent: {task.agent.role}")
    print(f"    Flow:  {status}")
    if i < len(tasks):
        print(f"    {'':8}|")
print("=" * 65)
print(f"\nTotal tasks: {len(tasks)} | Process: Sequential")
print("Each task's output becomes the next task's input context.")

## 5. Assemble and Run the Crew

### Crew Coordination Deep Dive

The `Crew` orchestrator handles:

| Responsibility | How It Works |
|---------------|-------------|
| **Task assignment** | Each task specifies its agent; the crew enforces this mapping |
| **Context passing** | Output of Task N is injected into Task N+1's prompt as context |
| **Execution order** | `Process.sequential` runs tasks in list order, one at a time |
| **Error handling** | If an agent produces poor output, downstream agents must work with it |
| **Verbose logging** | Shows each agent's chain-of-thought reasoning in real time |

**Why sequential over hierarchical?** Content creation is inherently a pipeline — research must happen before writing, writing before editing, etc. A hierarchical process (with a manager agent) adds overhead without benefit for linear workflows.

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,  # Set True for multi-run sessions to accumulate knowledge
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")
print(f"Memory: {'Enabled' if crew.memory else 'Disabled'}")

In [ ]:
# Execute the content pipeline
print("=" * 70)
print(f"LAUNCHING CONTENT CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Topic: {TOPIC}")
print(f"Pipeline: Research -> Write -> Edit -> SEO -> Repurpose")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Output — Repurposed Content Pack

The crew's final result is the repurposer's output: a multi-format content pack.

In [ ]:
print("=" * 70)
print("FINAL OUTPUT — Multi-Format Content Pack")
print("=" * 70)
print(result.raw)

### 6.2 Individual Agent Outputs

Each agent's deliverable is preserved separately — inspect the full delegation chain.

In [ ]:
for name, task in zip(task_names, tasks):
    print("\n" + "=" * 70)
    print(f"AGENT OUTPUT: {name}")
    print(f"Role: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2000:
            print(text[:2000])
            print(f"\n... [{len(text) - 2000} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 6.3 Pipeline Performance Metrics

In [ ]:
# Output size analysis
print("CONTENT PIPELINE METRICS")
print("=" * 55)
print(f"{'Stage':<20} {'Agent':<30} {'Output (chars)':>12}")
print("-" * 55)

total_chars = 0
for name, task in zip(task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total_chars += length
    print(f"{name:<20} {task.agent.role:<30} {length:>10,}")

print("-" * 55)
print(f"{'TOTAL':<50} {total_chars:>10,}")

# Token usage if available
if hasattr(result, 'token_usage') and result.token_usage:
    usage = result.token_usage
    print(f"\nTokens used: {usage.get('total_tokens', 'N/A')}")
else:
    print("\nToken usage: not tracked (typical with local Ollama)")

## 7. Save All Outputs

In [ ]:
# Save each agent's output and the combined report
outputs = {}
for name, task in zip(task_names, tasks):
    if task.output:
        outputs[name] = {
            "agent": task.agent.role,
            "output": task.output.raw,
            "length": len(task.output.raw),
        }

# Save combined report
report_lines = [
    f"# Content Pipeline Report",
    f"**Topic:** {TOPIC}",
    f"**Generated:** {datetime.now().isoformat()}",
    f"**Audience:** {AUDIENCE}",
    "---",
]

for name in task_names:
    if name in outputs:
        report_lines.append(f"\n## {name} (by {outputs[name]['agent']})\n")
        report_lines.append(outputs[name]["output"])
        report_lines.append("\n---")

report = "\n".join(report_lines)

with open("content_pipeline_report.md", "w", encoding="utf-8") as f:
    f.write(report)

# Save repurposed pieces separately
if "Repurpose" in outputs:
    with open("repurposed_content.md", "w", encoding="utf-8") as f:
        f.write(outputs["Repurpose"]["output"])

print(f"Reports saved:")
print(f"  content_pipeline_report.md ({len(report):,} chars)")
if "Repurpose" in outputs:
    print(f"  repurposed_content.md ({outputs['Repurpose']['length']:,} chars)")

## 8. Experiment: Different Topic

The entire pipeline is reusable — change the topic and run again to produce a full content package on any subject.

In [ ]:
# =====================================================
# SECOND CONTENT TOPIC
# =====================================================
TOPIC_2 = "Why Most AI Projects Fail: Lessons from the Graveyard of Enterprise AI Initiatives"
AUDIENCE_2 = "Business leaders, product managers, and data science team leads"
TONE_2 = "Frank, experience-driven, backed by failure case studies"
TARGET_KEYWORDS_2 = ["AI project failure", "enterprise AI", "AI implementation", "MLOps", "AI ROI"]

print(f"Second Topic: {TOPIC_2}")
print(f"Audience: {AUDIENCE_2}")
print(f"Keywords: {', '.join(TARGET_KEYWORDS_2)}")

In [ ]:
# Build new task set for the second topic
task2_research = Task(
    description=(
        f"Research this topic thoroughly: {TOPIC_2}\n"
        f"Audience: {AUDIENCE_2}\n\n"
        "Cover: key failure statistics, common failure modes, case studies (both "
        "failures and recoveries), expert quotes, and contrarian views."
    ),
    expected_output="Structured research brief with facts, cases, and angles.",
    agent=researcher,
)

task2_write = Task(
    description=(
        f"Write a 1500-2000 word blog post on: {TOPIC_2}\n"
        f"Audience: {AUDIENCE_2} | Tone: {TONE_2}\n"
        f"Keywords: {', '.join(TARGET_KEYWORDS_2)}\n\n"
        "Include a hook, real failure examples, data points, and actionable takeaways."
    ),
    expected_output="Complete blog post in markdown, 1500-2000 words.",
    agent=writer,
)

task2_edit = Task(
    description=(
        "Edit the blog post for clarity, grammar, flow, and factual accuracy. "
        "Return the FULL edited article with all changes applied."
    ),
    expected_output="Polished, publication-ready article.",
    agent=editor,
)

task2_seo = Task(
    description=(
        f"SEO audit for keywords: {', '.join(TARGET_KEYWORDS_2)}\n"
        "Check title tag, meta description, heading hierarchy, keyword density, "
        "and produce a scorecard rating each dimension 1-10."
    ),
    expected_output="SEO-optimized article + scorecard table.",
    agent=seo_checker,
)

task2_repurpose = Task(
    description=(
        "Repurpose the article into: (1) Twitter/X thread (8-12 tweets), "
        "(2) LinkedIn post (300 words with hashtags), (3) Email newsletter "
        "with subject line and CTA."
    ),
    expected_output="Three platform-specific content pieces.",
    agent=repurposer,
)

tasks_2 = [task2_research, task2_write, task2_edit, task2_seo, task2_repurpose]
print(f"Second topic tasks created: {len(tasks_2)} tasks")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Topic: {TOPIC_2}")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print(f"REPURPOSED CONTENT — {TOPIC_2[:50]}...")
print("=" * 70)
print(result2.raw)

## 9. Compare Both Runs

In [ ]:
print("PIPELINE COMPARISON")
print("=" * 70)
print(f"{'Stage':<18} {'SLMs (chars)':<18} {'AI Failures (chars)':<18}")
print("-" * 70)

for name, t1, t2 in zip(task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{name:<18} {len1:<18,} {len2:<18,}")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 70)
print(f"{'TOTAL':<18} {total1:<18,} {total2:<18,}")

## 10. Task Delegation Patterns Explained

### Pattern 1: Linear Chain (Used in This Notebook)

```
A -> B -> C -> D -> E
```

Each agent receives only the previous agent's output. Simple, predictable, and efficient.

**Pros:** Easy to debug, clear ownership, deterministic flow  
**Cons:** No feedback loops, one bad output propagates forward

### Pattern 2: Fan-Out After Editing

If tasks are independent (SEO and repurposing don't depend on each other), you could run them in parallel:

```
Research -> Write -> Edit -+-> SEO Audit
                           +-> Repurpose
```

CrewAI supports this via `context` parameter on tasks — you can point multiple tasks at the same upstream output.

### Pattern 3: Hierarchical with Manager

```
        Manager Agent
       /    |    |    \
  Research Writer Editor SEO
```

A manager agent decides which agent to invoke and when. Better for ambiguous tasks where the optimal order isn't known in advance.

### Pattern 4: Critic Loop (Iterative Refinement)

```
Write -> Edit -> Critic --[rejected]--> Write (retry)
                       --[approved]--> SEO -> Repurpose
```

Adding a critic agent that can send work back for revision. CrewAI supports this via custom task callbacks.

### Choosing the Right Pattern

| Pattern | Best For | Complexity |
|---------|----------|-----------|
| Linear Chain | Predictable pipelines | Low |
| Fan-Out | Independent parallel tasks | Medium |
| Hierarchical | Ambiguous or dynamic workflows | High |
| Critic Loop | Quality-critical content | High |

## 11. Advanced: Explicit Context Delegation

CrewAI allows explicit control over which task outputs each agent sees via the `context` parameter. This enables non-linear delegation patterns.

In [ ]:
# Demonstrate fan-out: SEO and Repurpose both receive the edited article directly

task_fan_research = Task(
    description=f"Research: {TOPIC}. Provide key facts, stats, and angles.",
    expected_output="Research brief.",
    agent=researcher,
)

task_fan_write = Task(
    description="Write a 1500-word blog post using the research.",
    expected_output="Blog post draft.",
    agent=writer,
    context=[task_fan_research],  # Explicitly takes research as input
)

task_fan_edit = Task(
    description="Edit the draft for clarity and quality.",
    expected_output="Edited article.",
    agent=editor,
    context=[task_fan_write],  # Takes the draft as input
)

# Fan-out: BOTH tasks receive the edited article
task_fan_seo = Task(
    description=f"SEO audit for keywords: {', '.join(TARGET_KEYWORDS)}",
    expected_output="SEO scorecard and optimized article.",
    agent=seo_checker,
    context=[task_fan_edit],  # Takes edited article
)

task_fan_repurpose = Task(
    description="Repurpose into Twitter thread, LinkedIn post, and newsletter.",
    expected_output="Three platform-specific content pieces.",
    agent=repurposer,
    context=[task_fan_edit],  # ALSO takes edited article (not SEO output)
)

fan_tasks = [task_fan_research, task_fan_write, task_fan_edit, task_fan_seo, task_fan_repurpose]

print("FAN-OUT DELEGATION PATTERN")
print("=" * 55)
print("  Research -> Write -> Edit -+-> SEO Audit")
print("                            +-> Repurpose")
print()
print("Both SEO and Repurpose receive the edited article directly.")
print("Neither depends on the other's output.")
print(f"\nTasks with explicit context: {sum(1 for t in fan_tasks if t.context)}/5")

In [ ]:
# Run the fan-out crew
crew_fan = Crew(
    agents=agents,
    tasks=fan_tasks,
    process=Process.sequential,  # Still sequential, but context controls data flow
    verbose=True,
    memory=False,
)

print(f"Launching fan-out crew — {datetime.now().strftime('%H:%M:%S')}")
result_fan = crew_fan.kickoff()
print(f"\nFan-out crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# Compare outputs — the repurposer worked from the edited article, not SEO version
fan_task_names = ["Research", "Write", "Edit", "SEO Audit", "Repurpose"]

print("FAN-OUT RESULTS")
print("=" * 55)
for name, task in zip(fan_task_names, fan_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx = ", ".join(t.agent.role for t in (task.context or []))
    print(f"  {name:<15} {length:>6,} chars | Context from: {ctx or 'auto (previous task)'}")

## 12. Key Takeaways

### What We Built
- A **5-agent content pipeline** (Researcher → Writer → Editor → SEO Checker → Repurposer)
- Ran it on **two different topics** to demonstrate reusability
- Demonstrated **fan-out delegation** where SEO and repurposing happen from the same source

### Task Delegation Concepts
1. **Sequential delegation** — each task's output automatically feeds the next
2. **Explicit context** — the `context` parameter gives fine-grained control over data flow
3. **Fan-out pattern** — multiple tasks can consume the same upstream output
4. **Agent specialization** — narrow roles produce better output than generalist agents

### Agent Design Lessons
- **Researcher backstory** emphasizes structured output → downstream agents get clean inputs
- **Writer backstory** specifies markdown formatting → editor gets consistent structure
- **Editor returns full text** (not suggestions) → SEO agent gets a complete article
- **SEO produces a scorecard** → measurable quality signal separate from the content
- **Repurposer restructures** (not truncates) → platform-native content, not summaries

### Production Tips
- Add `tools` (web search, file reading) to the researcher for real data access
- Use `output_pydantic` for structured, parseable agent outputs
- Enable `memory=True` for crews that produce content regularly (learns style over time)
- Consider adding a **Fact Checker** agent between Edit and SEO for high-stakes content
- Use `max_rpm` to rate-limit API calls when using hosted LLMs